# Lab 006 — Name the mechanism, measure the bias

**Lesson:** [`lessons/0006-missingness-taxonomy.html`](../lessons/0006-missingness-taxonomy.html) · **Phase / Year:** Phase 1 / Year 1

**Skill you are practising:** classify missingness as MCAR / MAR / MNAR by *what it depends on*, and choose handling that doesn't bias the baseline.

**Exit criteria:** the EXIT TICKET cell prints the three observed-vs-true means (MCAR ≈ unbiased; MAR & MNAR biased) and the two CV scores showing whether `add_indicator` recovered signal, plus your one-sentence cause.

---

### How this notebook works
- **PROVIDED** cells are complete — just run them.
- **TODO** cells have blanks (`____`). Fill them in.
- **CHECK** cells auto-run and give you immediate feedback — don't edit them.
- Run top to bottom. When the **EXIT TICKET** prints cleanly, paste it back to your teacher (or say *"lab done"*).

### Environment
One-time setup from the repo root: `bash labs/setup-env.sh`  
Then select kernel **Relational Labs (.venv)** or interpreter `.venv/bin/python`.

## Setup — PROVIDED (run me)

In [1]:
# PROVIDED
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

RNG = np.random.RandomState(0)
print("setup ok")

setup ok


## Data — PROVIDED

A clean dataset of `n` rows. `x2` is a fully observed driver; `x1` **correlates with `x2`** and is the column we will knock holes in. The label `y` depends on both. We record the *true* mean of `x1` before any holes appear.

In [2]:
# PROVIDED
n = 2000
x2 = RNG.normal(0, 1, n)                       # fully observed driver
x1 = 0.7 * x2 + RNG.normal(0, 0.7, n)          # correlates with x2 (~0.7)
logit = 0.9 * x1 + 0.9 * x2
y = (RNG.rand(n) < 1 / (1 + np.exp(-logit))).astype(int)

df = pd.DataFrame({"x1": x1, "x2": x2, "y": y})
TRUE_MEAN_X1 = df["x1"].mean()
print(f"n={n}  corr(x1,x2)={np.corrcoef(x1, x2)[0,1]:.2f}  true mean x1={TRUE_MEAN_X1:+.3f}")

n=2000  corr(x1,x2)=0.70  true mean x1=-0.036


## Task 1 — MCAR mask — PROVIDED (worked example)

MCAR = the probability of being missing depends on **nothing**. Study this; Tasks 2 and 3 follow the same shape.

In [3]:
# PROVIDED EXAMPLE — MCAR: independent of x1, x2, y.
mcar_mask = RNG.rand(n) < 0.40        # a fair coin per row
print("MCAR fraction missing:", round(mcar_mask.mean(), 3))

MCAR fraction missing: 0.391


## Task 2 — MAR mask — TODO

MAR = missingness depends on the **observed** column `x2`. Make `x1` more likely to be missing where `x2` is large.

Hint: a probability that rises with `x2` is `1 / (1 + np.exp(-(2 * x2)))`; then draw `RNG.rand(n) < prob`.

In [4]:
# TODO: fill the two blanks.
prob_mar = 1 / (1 + np.exp(-(2 * x2)))  # higher where x2 is large (use the hint)
mar_mask = RNG.rand(n) < prob_mar       # draw a boolean mask from prob_mar
print("MAR fraction missing:", round(mar_mask.mean(), 3))

MAR fraction missing: 0.5


In [5]:
# CHECK — do not edit.
assert getattr(mar_mask, "dtype", None) == bool and len(mar_mask) == n, "mar_mask must be a boolean array of length n."
corr_x2 = np.corrcoef(mar_mask.astype(float), x2)[0, 1]
corr_x1 = np.corrcoef(mar_mask.astype(float), x1)[0, 1]
assert corr_x2 > 0.20, f"MAR must depend on the OBSERVED x2 (corr={corr_x2:.2f}). Make missingness increase with x2."
print(f"Task 2 ok — missingness tracks observed x2 (corr={corr_x2:.2f}); incidental corr with x1={corr_x1:.2f}.")

Task 2 ok — missingness tracks observed x2 (corr=0.58); incidental corr with x1=0.39.


## Task 3 — MNAR mask — TODO

MNAR = missingness depends on the **value that goes missing** (`x1` itself). Make `x1` more likely to be missing where `x1` is large — the same shape as Task 2, but driven by `x1`.

In [6]:
# TODO: fill the two blanks.
prob_mnar = 1 / (1 + np.exp(-(2 * x1)))                   # higher where x1 is large
mnar_mask = RNG.rand(n) < prob_mnar                      # draw a boolean mask from prob_mnar
print("MNAR fraction missing:", round(mnar_mask.mean(), 3))

MNAR fraction missing: 0.487


In [7]:
# CHECK — do not edit.
assert getattr(mnar_mask, "dtype", None) == bool and len(mnar_mask) == n, "mnar_mask must be a boolean array of length n."
corr_self = np.corrcoef(mnar_mask.astype(float), x1)[0, 1]
assert corr_self > 0.20, f"MNAR must depend on x1 ITSELF (corr={corr_self:.2f})."
print(f"Task 3 ok — missingness tracks the hidden x1 itself (corr={corr_self:.2f}).")

Task 3 ok — missingness tracks the hidden x1 itself (corr=0.59).


## Task 4 — Observed-only mean vs truth — TODO

For each mechanism, compute the mean of `x1` over the rows where it is **not** missing, and compare to `TRUE_MEAN_X1`. Which mechanisms bias the naive mean?

In [8]:
def observed_mean(mask):
    """Mean of x1 over rows where it is NOT missing."""
    return x1[~mask].mean()

# TODO: fill the three blanks using observed_mean(...)
mean_mcar = observed_mean(mcar_mask)
mean_mar  = observed_mean(mar_mask)
mean_mnar = observed_mean(mnar_mask)

for name, m in [("MCAR", mean_mcar), ("MAR", mean_mar), ("MNAR", mean_mnar)]:
    print(f"{name:5s} observed mean {m:+.3f}   bias {m - TRUE_MEAN_X1:+.3f}")

MCAR  observed mean -0.034   bias +0.002
MAR   observed mean -0.415   bias -0.380
MNAR  observed mean -0.588   bias -0.552


In [9]:
# CHECK — do not edit.
for nm, v in [("mean_mcar", mean_mcar), ("mean_mar", mean_mar), ("mean_mnar", mean_mnar)]:
    assert v is not None and np.isfinite(v), f"{nm} not filled in."
assert abs(mean_mcar - TRUE_MEAN_X1) < 0.06, "MCAR mean should be ~unbiased — re-check mean_mcar."
assert abs(mean_mnar - TRUE_MEAN_X1) > abs(mean_mcar - TRUE_MEAN_X1), "MNAR should be MORE biased than MCAR."
print("Task 4 ok — MCAR ~unbiased; MAR & MNAR pulled away from the truth.")

Task 4 ok — MCAR ~unbiased; MAR & MNAR pulled away from the truth.


## Task 5 — Does the missing-indicator help? — TODO

On the **MNAR** data, build two Lesson-005 pipelines and compare 5-fold CV accuracy:
- `pipe_plain`: median-impute `x1`, **no** indicator.
- `pipe_ind`: median-impute `x1`, **with** `add_indicator=True`.

Because the gap encodes “`x1` was high,” the indicator should let the model recover signal that mean-imputation destroyed.

In [10]:
# PROVIDED — the MNAR design matrix (x1 has holes; x2 fully observed).
X_mnar = df[["x1", "x2"]].copy()
X_mnar.loc[mnar_mask, "x1"] = np.nan
print("missing in X_mnar:\n", X_mnar.isna().sum())

missing in X_mnar:
 x1    974
x2      0
dtype: int64


In [15]:
# TODO: fill the blanks (strategy and add_indicator settings).
pipe_plain = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=False)),   # median, no indicator
    ("clf",    LogisticRegression(max_iter=1000)),
])
pipe_ind = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),  # median + indicator
    ("clf",    LogisticRegression(max_iter=1000)),
])

score_plain = cross_val_score(pipe_plain, X_mnar, df["y"], cv=5).mean()
score_ind   = cross_val_score(pipe_ind,   X_mnar, df["y"], cv=5).mean()
print(f"plain impute     : {score_plain:.3f}")
print(f"impute+indicator : {score_ind:.3f}")
print(f"delta            : {score_ind - score_plain:+.3f}")

plain impute     : 0.721
impute+indicator : 0.737
delta            : +0.016


In [16]:
# CHECK — do not edit.
assert 0.5 <= score_plain <= 1.0 and 0.5 <= score_ind <= 1.0, "Scores should be valid accuracies (x2 alone carries signal)."
if score_ind > score_plain:
    print(f"Task 5 ok — the indicator recovered MNAR signal (+{score_ind - score_plain:.3f}).")
else:
    print(f"Task 5 ran. Indicator delta {score_ind - score_plain:+.3f} — small here; note why in your takeaway.")

Task 5 ok — the indicator recovered MNAR signal (+0.016).


## Exit ticket — TODO

Fill the one-sentence takeaway, run, and paste the output back to your teacher.

In [17]:
# TODO: complete the takeaway sentence.
print("=== EXIT TICKET — Lesson 006 ===")
print(f"true mean x1            : {TRUE_MEAN_X1:+.3f}")
print(f"observed mean | MCAR    : {mean_mcar:+.3f}  (bias {mean_mcar - TRUE_MEAN_X1:+.3f})")
print(f"observed mean | MAR     : {mean_mar:+.3f}  (bias {mean_mar - TRUE_MEAN_X1:+.3f})")
print(f"observed mean | MNAR    : {mean_mnar:+.3f}  (bias {mean_mnar - TRUE_MEAN_X1:+.3f})")
print(f"CV acc plain impute     : {score_plain:.3f}")
print(f"CV acc impute+indicator : {score_ind:.3f}")
print()
print("takeaway:", "MNAR biased mean, indicator helped to mitigate problem")   # which mechanisms biased the mean, and did the indicator help?

=== EXIT TICKET — Lesson 006 ===
true mean x1            : -0.036
observed mean | MCAR    : -0.034  (bias +0.002)
observed mean | MAR     : -0.415  (bias -0.380)
observed mean | MNAR    : -0.588  (bias -0.552)
CV acc plain impute     : 0.721
CV acc impute+indicator : 0.737

takeaway: MNAR biased mean, indicator helped to mitigate problem
